In [1]:
# Vérifier le chemin du dataset
import os

print("📂 Contenu de /kaggle/input/ :\n")
for root, dirs, files in os.walk('/kaggle/input/'):
    level = root.replace('/kaggle/input/', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"{indent}  📄 {f}")

📂 Contenu de /kaggle/input/ :

📁 /
📁 datasets/
  📁 zeinebbayoudh7/
    📁 dataset-voix/
      📁 final/
        📁 val/
          📁 grunt/
          📁 whine/
          📁 growl/
          📁 bark/
          📁 distress/
        📁 test/
          📁 grunt/
          📁 whine/
          📁 growl/
          📁 bark/
          📁 distress/
        📁 train/
          📁 grunt/
          📁 whine/
          📁 growl/
          📁 bark/
          📁 distress/


In [1]:
import os
print(os.listdir("/kaggle/working"))

['labels.csv', '.virtual_documents', 'resnet50_phase1_best.pth', 'efficientnet_phase2_best.pth', 'resnet50_phase2_best.pth', 'state.db', 'efficientnet_phase1_best.pth']


In [2]:
from IPython.display import FileLink
display(FileLink('efficientnet_phase1_best.pth'))
display(FileLink('efficientnet_phase2_best.pth'))
display(FileLink('resnet50_phase1_best.pth'))
display(FileLink('resnet50_phase2_best.pth'))

/kaggle/working/efficientnet_phase1_best.pth

/kaggle/working/efficientnet_phase2_best.pth

/kaggle/working/resnet50_phase1_best.pth

/kaggle/working/resnet50_phase2_best.pth

# importation des bib avec l'initialisation des param

In [4]:
# ============================================================
# CELLULE 1 — Imports + Paramètres
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

# Vérification GPU
if torch.cuda.is_available():
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  CPU uniquement")

# Chemins
DATA_DIR = '/kaggle/input/datasets/zeinebbayoudh7/dataset-voix/final'
CSV_PATH = '/kaggle/working/labels.csv'

# Paramètres audio
SR         = 22050
DURATION   = 4
N_MELS     = 128    # nombre de bandes mel
HOP_LENGTH = 512    # pas entre chaque frame
N_FFT      = 2048   # taille de la fenêtre FFT

# Paramètres training
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n📁 Data dir : {DATA_DIR}")
print(f"📁 CSV path : {CSV_PATH}")

✅ GPU : Tesla T4
   Mémoire : 14.6 GB

📁 Data dir : /kaggle/input/datasets/zeinebbayoudh7/dataset-voix/final
📁 CSV path : /kaggle/working/labels.csv


In [7]:
# ============================================================
# CELLULE 2 — Recréation labels.csv
# ============================================================
import pandas as pd

label_to_groupe = {
    'bark'     : 'normal',
    'grunt'    : 'normal',
    'whine'    : 'normal',
    'growl'    : 'anormal',
    'distress' : 'anormal',
}

records = []

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    for label in os.listdir(split_dir):
        label_dir = os.path.join(split_dir, label)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith('.wav'):
                records.append({
                    'filepath' : os.path.join(split, label, fname),
                    'label'    : label,
                    'groupe'   : label_to_groupe.get(label, 'inconnu'),
                    'split'    : split
                })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print(f"✅ labels.csv recréé : {len(df)} lignes\n")
print("📊 Distribution par groupe :")
print(df.groupby(['split', 'groupe']).size().unstack(fill_value=0))
print("\n📊 Distribution par label :")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

✅ labels.csv recréé : 790 lignes

📊 Distribution par groupe :
groupe  anormal  normal
split                  
test         50      75
train       220     330
val          46      69

📊 Distribution par label :
label  bark  distress  growl  grunt  whine
split                                     
test     25        25     25     25     25
train   110       110    110    110    110
val      23        23     23     23     23


Notre mapping normal/anormal est basé sur la littérature vétérinaire scientifique. Le bark, grunt et whine sont des vocalizations normales de communication. Le growl indique peur ou agressivité, et le distress (howl + yelp) indique une détresse sévère nécessitant une intervention vétérinaire ,ces 2 classes sont donc classifiées comme anormales dans notre système de détection

## source: 
Texas A&M Veterinary Medicine (2024)
    vetmed.tamu.edu/news/pet-talk/dog-vocalizations

WagWalking — Excessive Vocalization in Dogs
    wagwalking.com/condition/excessive-vocalization

Wisdom Panel (2025)
    wisdompanel.com/en-us/blog/why-do-dogs-bark

## Convertir Wav en mel spectrogram
Un Mel Spectrogram est une représentation visuelle d'un signal audio ,il convertit le son en image où l'axe horizontal représente le temps, l'axe vertical les fréquences, et la couleur l'intensité du son.
Mel: echelle ili ynajem yasm3ou humain , bech na3mlou compression lil son pour devient plus naturelle 

## remarque :
Mel brut :valeurs énormes (0.001 à 1,000,000) -> power_to_db

Décibels  → valeurs raisonnables (-80 à 0): normalisation

Image     → pixels (0 à 255) 

CNN analyse l'image comme une photo normale

In [5]:
# ============================================================
# CELLULE 3 — Fonction WAV → Mel Spectrogram
# ============================================================
import librosa
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

def wav_to_melspec(filepath, sr=22050, duration=4,
                   n_mels=128, hop_length=512, n_fft=2048):
 
    # Charger le fichier audio
    audio, sr = librosa.load(filepath, sr=sr, duration=duration)

    # Padding si le fichier est trop court
    target_len = sr * duration
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))

    # Créer le Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=n_fft
    )

    # Convertir en décibels (plus lisible pour le CNN)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normaliser entre 0 et 255 (comme une image)
    mel_norm = ((mel_db - mel_db.min()) /
                (mel_db.max() - mel_db.min()) * 255).astype(np.uint8)

    # Convertir en image RGB 224×224
    img = Image.fromarray(mel_norm).convert('RGB')
    img = img.resize((224, 224))

    return img

# Test sur un fichier
test_file = os.path.join(DATA_DIR, 'train', 'bark',
            os.listdir(os.path.join(DATA_DIR, 'train', 'bark'))[0])

img = wav_to_melspec(test_file)
print(f"✅ Mel Spectrogram créé !")
print(f"   Taille : {img.size}")
print(f"   Mode   : {img.mode}")

✅ Mel Spectrogram créé !
   Taille : (224, 224)
   Mode   : RGB


In [8]:
# ============================================================
# CELLULE 4 — Dataset PyTorch
# ============================================================
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Labels
LABELS   = sorted(df['label'].unique().tolist())
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print(f" {len(LABELS)} labels :")
for i, label in enumerate(LABELS):
    groupe = df[df['label'] == label]['groupe'].iloc[0]
    print(f"  {i} → {label:15s} ({groupe})")

# Transformations
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class AudioDataset(Dataset):
    def __init__(self, df, data_dir, split, transform=None):
        self.df        = df[df['split'] == split].reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.data_dir, row['filepath'])
        label = LABEL2ID[row['label']]

        # Convertir WAV en Mel Spectrogram
        img = wav_to_melspec(path)

        if self.transform:
            img = self.transform(img)
        return img, label

# Créer les datasets
train_dataset = AudioDataset(df, DATA_DIR, 'train', train_transform)
val_dataset   = AudioDataset(df, DATA_DIR, 'val',   val_transform)
test_dataset  = AudioDataset(df, DATA_DIR, 'test',  val_transform)

# Créer les dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print(f"\n Datasets créés :")
print(f"   train : {len(train_dataset)} images")
print(f"   val   : {len(val_dataset)} images")
print(f"   test  : {len(test_dataset)} images")

 5 labels :
  0 → bark            (normal)
  1 → distress        (anormal)
  2 → growl           (anormal)
  3 → grunt           (normal)
  4 → whine           (normal)

 Datasets créés :
   train : 550 images
   val   : 115 images
   test  : 125 images


## EfficientNet 
Nous comparons 3 architectures pour la classification audio : EfficientNetB0 et ResNet50 sur Mel Spectrogram (même pipeline que le comportement), et YAMNet fine-tuné comme modèle spécialisé audio.
## C quoi YamNet 
Définition simple
YAMNet = Yet Another Mobile Network
C'est un modèle créé par Google qui reconnaît 521 types de sons différents.

In [9]:
# ============================================================
# CELLULE 5 — Création EfficientNetB0
# ============================================================
import torchvision.models as models
import torch.nn as nn

def create_efficientnet(num_classes=5, dropout=0.5):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')

    # Geler le backbone (Phase 1)
    for param in model.parameters():
        param.requires_grad = False

    # Remplacer le classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    return model

model_eff = create_efficientnet(num_classes=5).to(device)

total_params     = sum(p.numel() for p in model_eff.parameters())
trainable_params = sum(p.numel() for p in model_eff.parameters()
                       if p.requires_grad)

print("✅ EfficientNetB0 créé")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")
print(f"   Classes          : 5 (bark/distress/growl/grunt/whine)")
print(f"   Dropout          : 0.5")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 122MB/s] 


✅ EfficientNetB0 créé
   Total params     : 4,013,953
   Trainable params : 6,405
   Classes          : 5 (bark/distress/growl/grunt/whine)
   Dropout          : 0.5


In [7]:
# ============================================================
# CELLULE 6 — Fonction train_model + Early Stopping
# ============================================================
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

def train_model(model, train_loader, val_loader,
                epochs, lr, label_smoothing=0.1,
                weight_decay=1e-4, phase=1,
                patience=5, model_name='model'):

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

    best_val_acc     = 0.0
    patience_counter = 0
    history          = {'train_loss': [], 'train_acc': [],
                        'val_loss':   [], 'val_acc':   []}

    print(f"🚀 Phase {phase} — lr={lr} | epochs={epochs} | patience={patience}\n")

    for epoch in range(epochs):

        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item()
            preds          = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs     = model(imgs)
                loss        = criterion(outputs, labels)
                val_loss   += loss.item()
                preds       = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        # ── Métriques ──────────────────────────────────────
        t_loss = train_loss / len(train_loader)
        t_acc  = train_correct / train_total * 100
        v_loss = val_loss / len(val_loader)
        v_acc  = val_correct / val_total * 100

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        # ── Early Stopping ─────────────────────────────────
        if v_acc > best_val_acc:
            best_val_acc     = v_acc
            patience_counter = 0
            torch.save(model.state_dict(),
                       f'/kaggle/working/{model_name}_phase{phase}_best.pth')
            saved = "💾"
        else:
            patience_counter += 1
            saved = f"⏳ {patience_counter}/{patience}"

        scheduler.step()

        print(f"  Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {t_loss:.4f} Acc: {t_acc:.1f}% | "
              f"Val Loss: {v_loss:.4f} Acc: {v_acc:.1f}% {saved}")

        if patience_counter >= patience:
            print(f"\n⛔ Early stopping à epoch {epoch+1}")
            break

    print(f"\n✅ Phase {phase} terminée — "
          f"Meilleure Val Acc: {best_val_acc:.1f}%")
    return history

print("✅ Fonction train_model définie")

✅ Fonction train_model définie


In [8]:
# ============================================================
# CELLULE 7 — EfficientNet Phase 1
# ============================================================
history_eff_p1 = train_model(
    model        = model_eff,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 20,
    lr           = 0.001,
    phase        = 1,
    patience     = 5,
    model_name   = 'efficientnet'
)

🚀 Phase 1 — lr=0.001 | epochs=20 | patience=5

  Epoch  1/20 | Train Loss: 1.5188 Acc: 35.5% | Val Loss: 1.4490 Acc: 49.6% 💾
  Epoch  2/20 | Train Loss: 1.2637 Acc: 58.0% | Val Loss: 1.2810 Acc: 61.7% 💾
  Epoch  3/20 | Train Loss: 1.0795 Acc: 68.7% | Val Loss: 1.2100 Acc: 62.6% 💾
  Epoch  4/20 | Train Loss: 1.0737 Acc: 69.3% | Val Loss: 1.1155 Acc: 68.7% 💾
  Epoch  5/20 | Train Loss: 1.0035 Acc: 74.0% | Val Loss: 1.0564 Acc: 71.3% 💾
  Epoch  6/20 | Train Loss: 0.9892 Acc: 71.6% | Val Loss: 1.0399 Acc: 73.0% 💾
  Epoch  7/20 | Train Loss: 0.9462 Acc: 74.4% | Val Loss: 1.0404 Acc: 73.9% 💾
  Epoch  8/20 | Train Loss: 0.9676 Acc: 72.9% | Val Loss: 1.0252 Acc: 73.0% ⏳ 1/5
  Epoch  9/20 | Train Loss: 0.9269 Acc: 75.1% | Val Loss: 1.0158 Acc: 74.8% 💾
  Epoch 10/20 | Train Loss: 0.9583 Acc: 72.9% | Val Loss: 1.0164 Acc: 72.2% ⏳ 1/5
  Epoch 11/20 | Train Loss: 0.8968 Acc: 75.3% | Val Loss: 0.9892 Acc: 73.9% ⏳ 2/5
  Epoch 12/20 | Train Loss: 0.9095 Acc: 74.7% | Val Loss: 0.9906 Acc: 73.9% ⏳ 3/5
 

In [9]:
# ============================================================
# CELLULE 8 — EfficientNet Phase 2
# ============================================================

model_eff.load_state_dict(
    torch.load('/kaggle/working/efficientnet_phase1_best.pth'))

for param in model_eff.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model_eff.parameters()
                   if p.requires_grad)
print(f"✅ Backbone dégelé — {total_params:,} paramètres\n")

history_eff_p2 = train_model(
    model        = model_eff,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 30,
    lr           = 0.0001,
    phase        = 2,
    patience     = 7,
    model_name   = 'efficientnet'
)

✅ Backbone dégelé — 4,013,953 paramètres

🚀 Phase 2 — lr=0.0001 | epochs=30 | patience=7

  Epoch  1/30 | Train Loss: 0.8731 Acc: 76.7% | Val Loss: 0.8258 Acc: 81.7% 💾
  Epoch  2/30 | Train Loss: 0.7323 Acc: 85.1% | Val Loss: 0.7616 Acc: 84.3% 💾
  Epoch  3/30 | Train Loss: 0.7064 Acc: 88.2% | Val Loss: 0.7298 Acc: 83.5% ⏳ 1/7
  Epoch  4/30 | Train Loss: 0.6301 Acc: 91.1% | Val Loss: 0.6822 Acc: 87.8% 💾
  Epoch  5/30 | Train Loss: 0.6274 Acc: 91.8% | Val Loss: 0.6567 Acc: 89.6% 💾
  Epoch  6/30 | Train Loss: 0.5837 Acc: 93.6% | Val Loss: 0.6471 Acc: 90.4% 💾
  Epoch  7/30 | Train Loss: 0.5674 Acc: 94.7% | Val Loss: 0.6346 Acc: 91.3% 💾
  Epoch  8/30 | Train Loss: 0.5680 Acc: 94.5% | Val Loss: 0.6281 Acc: 89.6% ⏳ 1/7
  Epoch  9/30 | Train Loss: 0.5890 Acc: 94.4% | Val Loss: 0.6209 Acc: 89.6% ⏳ 2/7
  Epoch 10/30 | Train Loss: 0.5637 Acc: 95.8% | Val Loss: 0.6205 Acc: 90.4% ⏳ 3/7
  Epoch 11/30 | Train Loss: 0.5315 Acc: 96.7% | Val Loss: 0.6024 Acc: 91.3% ⏳ 4/7
  Epoch 12/30 | Train Loss: 0.54

## ResNet 

In [10]:
def create_resnet50(num_classes=5, dropout=0.5):
    model = models.resnet50(weights='IMAGENET1K_V1')
    
    for param in model.parameters():
        param.requires_grad = False
    
    in_features = model.fc.in_features
    
    # Tête plus régularisée
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),  # couche intermédiaire
        nn.ReLU(),
        nn.Dropout(p=dropout),
        nn.BatchNorm1d(256),           # normalisation
        nn.Linear(256, num_classes)
    )
    return model

# Recréer
model_res = create_resnet50(num_classes=5, dropout=0.5).to(device)
print("✅ ResNet50 recréé avec architecture corrigée")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s] 


✅ ResNet50 recréé avec architecture corrigée


In [33]:
history_res_p1 = train_model(
    model        = model_res,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 15,
    lr           = 0.001,
    weight_decay = 1e-3,
    phase        = 1,
    patience     = 5,
    model_name   = 'resnet50'
)

🚀 Phase 1 — lr=0.001 | epochs=15 | patience=5

  Epoch  1/15 | Train Loss: 1.3781 Acc: 45.1% | Val Loss: 1.3617 Acc: 55.7% 💾
  Epoch  2/15 | Train Loss: 1.1615 Acc: 61.1% | Val Loss: 1.1977 Acc: 63.5% 💾
  Epoch  3/15 | Train Loss: 1.0749 Acc: 68.2% | Val Loss: 1.1158 Acc: 69.6% 💾
  Epoch  4/15 | Train Loss: 1.0309 Acc: 71.6% | Val Loss: 1.0183 Acc: 75.7% 💾
  Epoch  5/15 | Train Loss: 1.0346 Acc: 72.2% | Val Loss: 0.9814 Acc: 75.7% ⏳ 1/5
  Epoch  6/15 | Train Loss: 0.9677 Acc: 74.5% | Val Loss: 0.9583 Acc: 78.3% 💾
  Epoch  7/15 | Train Loss: 0.9309 Acc: 77.3% | Val Loss: 0.9187 Acc: 78.3% ⏳ 1/5
  Epoch  8/15 | Train Loss: 0.9153 Acc: 76.9% | Val Loss: 0.8762 Acc: 83.5% 💾
  Epoch  9/15 | Train Loss: 0.9168 Acc: 78.0% | Val Loss: 0.8639 Acc: 80.9% ⏳ 1/5
  Epoch 10/15 | Train Loss: 0.8750 Acc: 79.6% | Val Loss: 0.8816 Acc: 80.9% ⏳ 2/5
  Epoch 11/15 | Train Loss: 0.8582 Acc: 79.5% | Val Loss: 0.8679 Acc: 79.1% ⏳ 3/5
  Epoch 12/15 | Train Loss: 0.8570 Acc: 79.1% | Val Loss: 0.8081 Acc: 83.5%

In [34]:
# ============================================================
# CELLULE — ResNet50 Phase 2
# ============================================================
model_res.load_state_dict(
    torch.load('/kaggle/working/resnet50_phase1_best.pth'))

for param in model_res.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model_res.parameters()
                   if p.requires_grad)
print(f"✅ Backbone dégelé — {total_params:,} paramètres\n")

history_res_p2 = train_model(
    model        = model_res,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 30,
    lr           = 0.0001,
    weight_decay = 1e-3,
    phase        = 2,
    patience     = 7,
    model_name   = 'resnet50'
)

✅ Backbone dégelé — 24,034,373 paramètres

🚀 Phase 2 — lr=0.0001 | epochs=30 | patience=7

  Epoch  1/30 | Train Loss: 0.8543 Acc: 79.6% | Val Loss: 0.6817 Acc: 92.2% 💾
  Epoch  2/30 | Train Loss: 0.7030 Acc: 89.5% | Val Loss: 0.6172 Acc: 93.0% 💾
  Epoch  3/30 | Train Loss: 0.6176 Acc: 94.2% | Val Loss: 0.6353 Acc: 92.2% ⏳ 1/7
  Epoch  4/30 | Train Loss: 0.5788 Acc: 95.1% | Val Loss: 0.5778 Acc: 94.8% 💾
  Epoch  5/30 | Train Loss: 0.5244 Acc: 97.5% | Val Loss: 0.5382 Acc: 93.9% ⏳ 1/7
  Epoch  6/30 | Train Loss: 0.5286 Acc: 98.4% | Val Loss: 0.5139 Acc: 94.8% ⏳ 2/7
  Epoch  7/30 | Train Loss: 0.5102 Acc: 96.9% | Val Loss: 0.5393 Acc: 94.8% ⏳ 3/7
  Epoch  8/30 | Train Loss: 0.5054 Acc: 98.4% | Val Loss: 0.5042 Acc: 96.5% 💾
  Epoch  9/30 | Train Loss: 0.4743 Acc: 98.7% | Val Loss: 0.5112 Acc: 93.9% ⏳ 1/7
  Epoch 10/30 | Train Loss: 0.4899 Acc: 99.3% | Val Loss: 0.5089 Acc: 94.8% ⏳ 2/7
  Epoch 11/30 | Train Loss: 0.4615 Acc: 99.3% | Val Loss: 0.4960 Acc: 95.7% ⏳ 3/7
  Epoch 12/30 | Train L

In [11]:
import os
for f in os.listdir('/kaggle/working/'):
    if f.endswith('.pth'):
        print(f"✅ {f}")

✅ resnet50_phase1_best.pth
✅ resnet50_phase2_best.pth
✅ efficientnet_phase1_best.pth
✅ efficientnet_phase2_best.pth


In [9]:
# ============================================================
# RECHARGEMENT CORRECT
# ============================================================

# EfficientNet
model_eff = create_efficientnet(num_classes=5).to(device)
model_eff.load_state_dict(
    torch.load('/kaggle/working/efficientnet_phase2_best.pth'))
model_eff.eval()
print("✅ EfficientNet rechargé")

# ResNet50 → nouvelle architecture avec dropout=0.5
model_res = create_resnet50(num_classes=5, dropout=0.5).to(device)
model_res.load_state_dict(
    torch.load('/kaggle/working/resnet50_phase2_best.pth'))
model_res.eval()
print("✅ ResNet50 rechargé")

✅ EfficientNet rechargé
✅ ResNet50 rechargé


In [10]:
# ============================================================
# ÉVALUATION — EfficientNet + ResNet sur Test Set
# ============================================================
from sklearn.metrics import classification_report

def evaluate_model(model, test_loader, model_name):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs    = imgs.to(device)
            outputs = model(imgs)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = sum(p == l for p, l in
              zip(all_preds, all_labels)) / len(all_labels) * 100

    print(f"\n{'='*50}")
    print(f"📊 {model_name} — Test Accuracy : {acc:.1f}%")
    print(f"{'='*50}")
    print(classification_report(all_labels, all_preds,
                                target_names=LABELS))
    return all_preds, all_labels

# Évaluation
preds_eff, labels_test = evaluate_model(
    model_eff, test_loader, 'EfficientNetB0')
preds_res, _           = evaluate_model(
    model_res, test_loader, 'ResNet50')


📊 EfficientNetB0 — Test Accuracy : 90.4%
              precision    recall  f1-score   support

        bark       0.96      0.92      0.94        25
    distress       0.95      0.72      0.82        25
       growl       0.96      0.96      0.96        25
       grunt       0.89      0.96      0.92        25
       whine       0.80      0.96      0.87        25

    accuracy                           0.90       125
   macro avg       0.91      0.90      0.90       125
weighted avg       0.91      0.90      0.90       125


📊 ResNet50 — Test Accuracy : 96.8%
              precision    recall  f1-score   support

        bark       0.92      0.96      0.94        25
    distress       0.96      0.88      0.92        25
       growl       1.00      1.00      1.00        25
       grunt       1.00      1.00      1.00        25
       whine       0.96      1.00      0.98        25

    accuracy                           0.97       125
   macro avg       0.97      0.97      0.97       125

ResNet50 obtient 96.8% de test accuracy contre 90.4% pour EfficientNetB0. La cohérence entre val (96.5%) et test (96.8%) confirme l'absence d'overfitting malgré un train accuracy élevé. La classe distress reste la plus difficile avec 88% de recall pour ResNet50 — ce qui est acceptable dans notre contexte car cette classe combine howl et yelp qui peuvent avoir des patterns audio différents

In [2]:
import pandas as pd
print(pd.read_csv("/kaggle/working/labels.csv")['label'].unique())

['grunt' 'whine' 'growl' 'bark' 'distress']


In [3]:
print(pd.read_csv("/kaggle/working/labels.csv")['label'].value_counts())
print(f"\nTotal : {len(pd.read_csv('/kaggle/working/labels.csv'))}")

label
grunt       158
whine       158
growl       158
bark        158
distress    158
Name: count, dtype: int64

Total : 790


In [12]:
checkpoint = torch.load("/kaggle/working/resnet50_phase2_best.pth")
print([k for k in checkpoint.keys() if 'fc' in k])

['fc.0.weight', 'fc.0.bias', 'fc.3.weight', 'fc.3.bias', 'fc.3.running_mean', 'fc.3.running_var', 'fc.3.num_batches_tracked', 'fc.4.weight', 'fc.4.bias']


In [14]:
checkpoint = torch.load("/kaggle/working/resnet50_phase2_best.pth")
for k in checkpoint.keys():
    if 'fc' in k:
        print(k, checkpoint[k].shape)

fc.0.weight torch.Size([256, 2048])
fc.0.bias torch.Size([256])
fc.3.weight torch.Size([256])
fc.3.bias torch.Size([256])
fc.3.running_mean torch.Size([256])
fc.3.running_var torch.Size([256])
fc.3.num_batches_tracked torch.Size([])
fc.4.weight torch.Size([5, 256])
fc.4.bias torch.Size([5])


In [15]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES_VOIX = 5

# Charger EfficientNetB0
model_eff = models.efficientnet_b0(weights=None)
model_eff.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_eff.classifier[1].in_features, NUM_CLASSES_VOIX)
)
model_eff.load_state_dict(torch.load("/kaggle/working/efficientnet_phase2_best.pth"))
model_eff = model_eff.to(device)
model_eff.eval()
print("✅ EfficientNetB0 chargé")

# Charger ResNet50
model_res = models.resnet50(weights=None)
model_res.fc = nn.Sequential(
    nn.Linear(model_res.fc.in_features, 256),  # fc.0
    nn.ReLU(),                                   # fc.1
    nn.ReLU(),                                   # fc.2
    nn.BatchNorm1d(256),                         # fc.3
    nn.Linear(256, NUM_CLASSES_VOIX)             # fc.4
)
model_res.load_state_dict(torch.load("/kaggle/working/resnet50_phase2_best.pth"))
model_res = model_res.to(device)
model_res.eval()
print("✅ ResNet50 chargé")

✅ EfficientNetB0 chargé
✅ ResNet50 chargé


In [16]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Labels
df = pd.read_csv("/kaggle/working/labels.csv")
test_df = df[df["split"] == "test"].reset_index(drop=True)

CLASSES = sorted(test_df['label'].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

NORMAL_CLASSES  = ['bark', 'grunt', 'whine']
ANORMAL_CLASSES = ['growl', 'distress']

print(f"Classes : {CLASSES}")
print(f"Test set : {len(test_df)} images")

Classes : ['bark', 'distress', 'growl', 'grunt', 'whine']
Test set : 125 images


In [26]:
import librosa
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch

class VoixDataset(Dataset):
    def __init__(self, data_dir, classes, transform):
        self.samples   = []
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(sorted(classes))}
        
        for label in classes:
            label_dir = f"{data_dir}/{label}"
            for f in os.listdir(label_dir):
                if f.endswith('.wav'):
                    self.samples.append((f"{label_dir}/{f}", label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        
        # Charger audio et convertir en spectrogramme
        y, sr = librosa.load(path, sr=22050, duration=3.0)
        mel   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Normaliser entre 0 et 255
        mel_norm = ((mel_db - mel_db.min()) / 
                   (mel_db.max() - mel_db.min()) * 255).astype(np.uint8)
        
        # Convertir en image RGB 224x224
        img = Image.fromarray(mel_norm).convert("RGB").resize((224, 224))
        
        return self.transform(img), self.class_to_idx[label]

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

CLASSES  = ['bark', 'distress', 'growl', 'grunt', 'whine']
DATA_DIR = "/kaggle/input/datasets/zeinebbayoudh7/dataset-voix/final/test"

test_dataset = VoixDataset(DATA_DIR, CLASSES, val_tf)
test_loader  = DataLoader(test_dataset, batch_size=32, 
                          shuffle=False, num_workers=2)

print(f"✅ Dataset voix : {len(test_dataset)} fichiers")

✅ Dataset voix : 125 fichiers


In [27]:
# Extraire probabilités EfficientNetB0
all_probs_eff = []
all_labels    = []

model_eff.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = torch.softmax(model_eff(imgs), dim=1)
        all_probs_eff.extend(out.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs_eff = np.array(all_probs_eff)
all_labels    = np.array(all_labels)
print(f"✅ EfficientNetB0 probs extraites : {all_probs_eff.shape}")

# Extraire probabilités ResNet50
all_probs_res = []

model_res.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = torch.softmax(model_res(imgs), dim=1)
        all_probs_res.extend(out.cpu().numpy())

all_probs_res = np.array(all_probs_res)
print(f"✅ ResNet50 probs extraites : {all_probs_res.shape}")

✅ EfficientNetB0 probs extraites : (125, 5)
✅ ResNet50 probs extraites : (125, 5)


In [28]:
# Fusion EfficientNetB0 + ResNet50
probs_fusion_voix = (0.5 * all_probs_eff + 0.5 * all_probs_res)

# Mapping normal/anormal
ANORMAL_CLASSES = ['growl', 'distress']
NORMAL_CLASSES  = ['bark', 'grunt', 'whine']

ANORMAL_IDX = [CLASSES.index(c) for c in ANORMAL_CLASSES]
NORMAL_IDX  = [CLASSES.index(c) for c in NORMAL_CLASSES]

# Score anormal
score_anormal = probs_fusion_voix[:, ANORMAL_IDX].sum(axis=1)

# Décision
predictions = (score_anormal >= 0.5).astype(int)

# Labels réels
true_labels = np.array([1 if CLASSES[l] in ANORMAL_CLASSES else 0 
                         for l in all_labels])

# Métriques
from sklearn.metrics import accuracy_score, recall_score, classification_report

acc    = accuracy_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)

print(f"✅ Late Fusion Voix")
print(f"Accuracy       : {acc*100:.1f}%")
print(f"Recall anormal : {recall*100:.1f}%")
print(f"\n{classification_report(true_labels, predictions, target_names=['normal','anormal'])}")

✅ Late Fusion Voix
Accuracy       : 92.8%
Recall anormal : 84.0%

              precision    recall  f1-score   support

      normal       0.90      0.99      0.94        75
     anormal       0.98      0.84      0.90        50

    accuracy                           0.93       125
   macro avg       0.94      0.91      0.92       125
weighted avg       0.93      0.93      0.93       125



In [29]:
np.save("/kaggle/working/voix_probs_fusion.npy", probs_fusion_voix)
np.save("/kaggle/working/voix_true_labels.npy", true_labels)
print("✅ Probabilités voix sauvegardées")

✅ Probabilités voix sauvegardées


In [30]:
import os, shutil

os.makedirs("/kaggle/working/late_fusion_data", exist_ok=True)

shutil.copy("/kaggle/working/voix_probs_fusion.npy", 
            "/kaggle/working/late_fusion_data/voix_probs_fusion.npy")
shutil.copy("/kaggle/working/voix_true_labels.npy", 
            "/kaggle/working/late_fusion_data/voix_true_labels.npy")

print(os.listdir("/kaggle/working/late_fusion_data"))
print("✅ Fichiers prêts")

['voix_true_labels.npy', 'voix_probs_fusion.npy']
✅ Fichiers prêts


In [1]:
import os
print(os.listdir("/kaggle/working"))

['labels.csv', '.virtual_documents', 'efficientnet_phase2_best.pth', 'resnet50_phase1_best.pth', 'voix_true_labels.npy', 'state.db', 'voix_probs_fusion.npy', 'late_fusion_data', 'efficientnet_phase1_best.pth', 'resnet50_phase2_best.pth']
